In [2]:
import time
import xarray as xr
import numpy as np
import s3fs
import cartopy.crs as ccrs
from matplotlib import pyplot as plt
import cartopy
import earthaccess
from earthaccess import Auth, DataCollections, DataGranules, Store
%matplotlib inline

#### Code from https://podaac.github.io/tutorials/notebooks/datasets/DirectCloud_Access_SWOT_Oceanography.html

In [1]:
auth = earthaccess.login()

NameError: name 'earthaccess' is not defined

In [3]:
def init_cartopy(projection, box, latstep, lonstep, land, zorder=4,**karg):
    import cartopy
    import matplotlib.ticker as mticker
    import cartopy.feature as cfeature
    from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

    ax=plt.subplot(1,1,1,projection=projection)    
    if land==True:
        land_10m = cfeature.NaturalEarthFeature('physical', 'land', '10m',facecolor=cfeature.COLORS['land'])
        ax.add_feature(land_10m, zorder=zorder)
    ax.add_feature(cartopy.feature.LAKES, alpha=0.5, zorder=zorder,edgecolor='black')
    ax.add_feature(cartopy.feature.RIVERS, zorder=zorder)
    # ax.set_extent([box[0], box[1], box[2]+0.5, box[3]-1], crs=cartopy.crs.PlateCarree())
    if 'pacific' in karg:
        ax.set_extent([box[1], box[0]+360, box[2], box[3]], crs=cartopy.crs.PlateCarree())
    else:
        ax.set_extent([box[0], box[1], box[2], box[3]], crs=cartopy.crs.PlateCarree())
    ax.coastlines('10m')
    gl = ax.gridlines(draw_labels=True, x_inline=False, y_inline=False)
    if 'pacific' in karg:
        gl.xlocator = mticker.FixedLocator(np.hstack((np.arange(box[1],180,lonstep),np.arange(-180+lonstep,box[0],lonstep))))
    else:
        gl.xlocator = mticker.FixedLocator(np.arange(box[0],box[1],lonstep))
    gl.xformatter = LONGITUDE_FORMATTER
    gl.xlabel_style = {'size': 20, 'color': 'k','rotation':0}
    gl.yformatter = LATITUDE_FORMATTER
    gl.ylocator = mticker.FixedLocator(np.arange(box[2], box[3],latstep))
    gl.ylabel_style = {'size': 20, 'color': 'k','rotation':0}
    gl.top_labels=False
    gl.right_labels=False
    
    return ax, gl


# Unsmoothed

## Access SWOT Level 2 KaRIn Low Rate Sea Surface Height Data Product Files on cloud

In [4]:
lonmin = 10
lonmax = 40
latmin = -45
latmax = -20

In [5]:
#retrieves granule from the day we want
#Version D: new version
#Basic: 2km
#Unsmoothed: 250m but separated between left and right swath (2 groups)
#Expert: just contains many many variables
karin_results_unsmoothed = earthaccess.search_data(short_name = 'SWOT_L2_LR_SSH_D',
                                        temporal = ("2024-10-01", "2024-10-11"),
                                        bounding_box = (lonmin,latmin,lonmax,latmax),
                                       granule_name = '*Unsmoothed*')

Granules found: 29


In [6]:
#for the unsmoothed, there are 2 groups left and right
#there are also other dimensions that have different size according to files so we drop it
def drop_doppler_vars(ds):
    bad_vars = [
        v for v in ds.data_vars
        if "num_doppler_miti_lines" in ds[v].dims
    ]
    ds = ds.drop_vars(bad_vars)
    return ds.drop_dims("num_doppler_miti_lines", errors="ignore")

In [7]:
#opens granules and load into xarray dataset
ds_unsmoothed_l = xr.open_mfdataset(earthaccess.open(karin_results_unsmoothed), group='left', combine='nested', concat_dim="num_lines", decode_times=False, 
    preprocess=drop_doppler_vars, engine='h5netcdf')
ds_unsmoothed_l

Opening 29 granules, approx size: 22.72 GB
using endpoint: https://archive.swot.podaac.earthdata.nasa.gov/s3credentials


QUEUEING TASKS | : 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 2452.81it/s]
PROCESSING TASKS | : 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 73.73it/s]
COLLECTING RESULTS | : 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 286874.57it/s]


<xarray.Dataset> Size: 89GB
Dimensions:                                (num_lines: 2370446, num_pixels: 240)
Coordinates:
    latitude                               (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    longitude                              (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/26)
    time                                   (num_lines) float64 19MB dask.array<chunksize=(81956,), meta=np.ndarray>
    time_tai                               (num_lines) float64 19MB dask.array<chunksize=(81956,), meta=np.ndarray>
    latitude_uncert                        (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    longitude_uncert                       (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    ssh_karin                              (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    ssh_karin_qual                         (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    ...                                     ...
    height_cor_xover_qual                  (num_lines, num_pixels) float32 2GB dask.array<chunksize=(27319, 80), meta=np.ndarray>
    sea_state_bias_cor                     (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    sea_state_bias_cor_2                   (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    volumetric_correlation                 (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    volumetric_correlation_uncert          (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    total_coherence                        (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
Attributes:
    description:  Unsmoothed SSH measurement data and related information for...

In [8]:
#opens granules and load into xarray dataset
ds_unsmoothed_r = xr.open_mfdataset(earthaccess.open(karin_results_unsmoothed), group='right', combine='nested', concat_dim="num_lines", decode_times=False, 
                                    preprocess=drop_doppler_vars, engine='h5netcdf')
ds_unsmoothed_r

Opening 29 granules, approx size: 22.72 GB
using endpoint: https://archive.swot.podaac.earthdata.nasa.gov/s3credentials


QUEUEING TASKS | : 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 5802.08it/s]
PROCESSING TASKS | : 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 238.91it/s]
COLLECTING RESULTS | : 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 29/29 [00:00<00:00, 259349.29it/s]


<xarray.Dataset> Size: 89GB
Dimensions:                                (num_lines: 2370446, num_pixels: 240)
Coordinates:
    latitude                               (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    longitude                              (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
Dimensions without coordinates: num_lines, num_pixels
Data variables: (12/26)
    time                                   (num_lines) float64 19MB dask.array<chunksize=(81956,), meta=np.ndarray>
    time_tai                               (num_lines) float64 19MB dask.array<chunksize=(81956,), meta=np.ndarray>
    latitude_uncert                        (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    longitude_uncert                       (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    ssh_karin                              (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    ssh_karin_qual                         (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    ...                                     ...
    height_cor_xover_qual                  (num_lines, num_pixels) float32 2GB dask.array<chunksize=(27319, 80), meta=np.ndarray>
    sea_state_bias_cor                     (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    sea_state_bias_cor_2                   (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
    volumetric_correlation                 (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    volumetric_correlation_uncert          (num_lines, num_pixels) float64 5GB dask.array<chunksize=(16392, 48), meta=np.ndarray>
    total_coherence                        (num_lines, num_pixels) float32 2GB dask.array<chunksize=(20489, 60), meta=np.ndarray>
Attributes:
    description:  Unsmoothed SSH measurement data and related information for...

## Cross Over Calibration and Corrections

In [5]:
ds_unsmoothed_l['ssha_karin_2_corrected'] = ds_unsmoothed_l.ssha_karin_2 + ds_unsmoothed_l.height_cor_xover
ds_unsmoothed_l.ssha_karin_2_corrected

ds_unsmoothed_r['ssha_karin_2_corrected'] = ds_unsmoothed_r.ssha_karin_2 + ds_unsmoothed_r.height_cor_xover
ds_unsmoothed_r.ssha_karin_2_corrected

NameError: name 'ds_unsmoothed_l' is not defined

In [4]:
#no rain flag in unsmoothed
flag_masks = [1,2,4,8,16,64,128,256,1024,2048,4096,8192,32768,65536,131072,262144,524288,16777216,33554432,67108864,536870912,1073741824,2147483648]
qual = ds_unsmoothed_l.ssha_karin_2_qual.astype("uint32")
bad_mask = xr.zeros_like(qual, dtype=bool)
for f in flag_masks:
    bad_mask = bad_mask | ((qual & np.uint32(f)) != 0)

good = (~bad_mask)
ds_unsmoothed_l['ssha_karin_2_corrected'] = ds_unsmoothed_l.ssha_karin_2_corrected.where(good)

qual = ds_unsmoothed_r.ssha_karin_2_qual.astype("uint32")
bad_mask = xr.zeros_like(qual, dtype=bool)
for f in flag_masks:
    bad_mask = bad_mask | ((qual & np.uint32(f)) != 0)

good = (~bad_mask)
ds_unsmoothed_r['ssha_karin_2_corrected'] = ds_unsmoothed_r.ssha_karin_2_corrected.where(good)

NameError: name 'ds_unsmoothed_l' is not defined

## Plot

In [11]:
box = [lonmin, lonmax, latmin, latmax]
latstep=10
lonstep=10
land=True

In [12]:
#Issue: 
#pcolormesh doesn't take masked array nor Nans in lat/lon so we are fixing that by replacing Nans in lat/lon by 
#neighbooring values and put corresponding ssh at Nans but we have to have different cases in case Nans are on the side 
#of the array or the whole row/column
def prep_swot_for_pcolormesh(lon, lat, ssh):
    """
    Prepare raw SWOT swath arrays for pcolormesh plotting in Matplotlib 3.8+.

    Parameters
    ----------
    lon : np.ndarray
        2D longitude array (along-track × cross-track)
    lat : np.ndarray
        2D latitude array (along-track × cross-track)
    ssh : np.ndarray
        2D SSH array (along-track × cross-track)

    Returns
    -------
    lon2, lat2, ssh2 : np.ndarray
        Arrays ready for pcolormesh:
        - lon2, lat2: finite everywhere, degenerate quads at gaps
        - ssh2: NaN where data is invalid
    """
    # Step 0: convert to plain numpy arrays
    lon = np.asarray(lon)
    lat = np.asarray(lat)
    ssh = np.asarray(ssh)

    # Step 1: compute validity mask
    valid = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(ssh)
    bad = ~valid

    # Step 2: copy arrays for plotting
    lon2 = lon.copy()
    lat2 = lat.copy()

    # Step 3: detect large jumps (swath edges, orbit gaps)
    jump = np.zeros_like(valid, dtype=bool)
    # cross-track direction
    jump[:, 1:] |= (np.abs(np.diff(lon, axis=1)) > 2.0) | (np.abs(np.diff(lat, axis=1)) > 2.0)
    # along-track direction
    jump[1:, :] |= (np.abs(np.diff(lon, axis=0)) > 2.0) | (np.abs(np.diff(lat, axis=0)) > 2.0)
    bad |= jump

    # Step 4: collapse bad points onto left neighbor (degenerate quads)
    lon2[bad] = np.roll(lon2, 1, axis=1)[bad]
    lat2[bad] = np.roll(lat2, 1, axis=1)[bad]

    # Step 5: mask SSH where bad
    ssh2 = np.where(~bad, ssh, np.nan)

    # Step 6: fill any remaining NaNs in lon/lat with nearest neighbor
    def fill_nan_geometry_only(arr):
        out = arr.copy()
        # Left → right
        for j in range(1, out.shape[1]):
            mask = ~np.isfinite(out[:, j])
            out[mask, j] = out[mask, j - 1]
        # Right → left
        for j in range(out.shape[1] - 2, -1, -1):
            mask = ~np.isfinite(out[:, j])
            out[mask, j] = out[mask, j + 1]
        # Top → bottom
        for i in range(1, out.shape[0]):
            mask = ~np.isfinite(out[i, :])
            out[i, mask] = out[i - 1, mask]
        # Bottom → top
        for i in range(out.shape[0] - 2, -1, -1):
            mask = ~np.isfinite(out[i, :])
            out[i, mask] = out[i + 1, mask]
        return out

    lon2 = fill_nan_geometry_only(lon2)
    lat2 = fill_nan_geometry_only(lat2)

    return lon2, lat2, ssh2

In [13]:
# Load arrays from xarray/dask
lon_l = ds_unsmoothed_l.longitude.compute().values
lat_l = ds_unsmoothed_l.latitude.compute().values
ssh_l = ds_unsmoothed_l.ssha_karin_2_corrected.compute().values

lon_r = ds_unsmoothed_r.longitude.compute().values
lat_r = ds_unsmoothed_r.latitude.compute().values
ssh_r = ds_unsmoothed_r.ssha_karin_2_corrected.compute().values

In [14]:
# Prepare for plotting
#lon_l2, lat_l2, ssh_l2 = prep_swot_for_pcolormesh(lon_l, lat_l, ssh_l)
#lon_r2, lat_r2, ssh_r2 = prep_swot_for_pcolormesh(lon_r, lat_r, ssh_r) 
#takes too much memory so other option below

#should be 0
#print("Non-finite lon_l2:", np.sum(~np.isfinite(lon_l2)))
#print("Non-finite lat_l2:", np.sum(~np.isfinite(lat_l2)))
#print("Non-finite lon_l2:", np.sum(~np.isfinite(lon_r2)))
#print("Non-finite lat_l2:", np.sum(~np.isfinite(lat_r2)))

In [15]:
#fig = plt.figure(figsize=(15,5))
#data_proj = cartopy.crs.PlateCarree(central_longitude=25)
#map_proj = cartopy.crs.PlateCarree(central_longitude=25)
#ax, gl = init_cartopy(map_proj, box, latstep, lonstep, land) 
#palette=plt.cm.coolwarm
#palette.set_bad('w',1.0)
#pp=ax.pcolormesh(lon_l2,lat_l2,ssh_l2, transform=ccrs.PlateCarree(),vmin =-1, vmax=1, cmap=palette,shading="auto")
#pt=ax.pcolormesh(lon_r2,lat_r2,ssh_r2, transform=ccrs.PlateCarree(),vmin =-1, vmax=1, cmap=palette,shading="auto")
#plt.subplots_adjust(right=0.85,left=0.01,top=0.99,bottom=0.1)
#cbar_ax = fig.add_axes([0.9, 0.1, 0.04, 0.7])
#h=plt.colorbar(pp, cax=cbar_ax,orientation='vertical')
#h.set_label('SSHA Karin unsmoothed corrected and flagged (m)',fontsize=14)
#h.ax.tick_params(labelsize=14)

In [ ]:
fig = plt.figure(figsize=(15,5))
data_proj = cartopy.crs.PlateCarree(central_longitude=25)
map_proj = cartopy.crs.PlateCarree(central_longitude=25)
ax, gl = init_cartopy(map_proj, box, latstep, lonstep, land) 
palette=plt.cm.coolwarm
palette.set_bad('w',1.0)

#to solve memory issues
chunk = 50_000
for i0 in range(0, lon_l.shape[0], chunk):
    i1 = min(i0 + chunk, lon_l.shape[0])

    lon_c = lon_l[i0:i1, :]
    lat_c = lat_l[i0:i1, :]
    ssh_c = ssh_l[i0:i1, :]
    lon_l2, lat_l2, ssh_l2 = prep_swot_for_pcolormesh(lon_c, lat_c, ssh_c)
    #chunk reintroduces infinite values so we get rid of rows
    good_rows = np.all(np.isfinite(lon_l2) & np.isfinite(lat_l2), axis=1)
    lon_l2 = lon_l2[good_rows]
    lat_l2 = lat_l2[good_rows]
    ssh_l2 = ssh_l2[good_rows]
    
    if ssh_l2.shape[0] == 0:
        continue
        
    pp=ax.pcolormesh(
        lon_l2, lat_l2, ssh_l2,
        transform=ccrs.PlateCarree(),
        cmap=palette,
        vmin=-1, vmax=1,
        shading="auto")

for i0 in range(0, lon_r.shape[0], chunk):
    i1 = min(i0 + chunk, lon_r.shape[0])
    lon_c = lon_r[i0:i1, :]
    lat_c = lat_r[i0:i1, :]
    ssh_c = ssh_r[i0:i1, :]
    lon_r2, lat_r2, ssh_r2 = prep_swot_for_pcolormesh(lon_c, lat_c, ssh_c)
    #chunk reintroduces infinite values so we get rid of rows
    good_rows = np.all(np.isfinite(lon_r2) & np.isfinite(lat_r2), axis=1)
    lon_r2 = lon_r2[good_rows]
    lat_r2 = lat_r2[good_rows]
    ssh_r2 = ssh_r2[good_rows]
    
    if ssh_r2.shape[0] == 0:
        continue

    pt=ax.pcolormesh(
        lon_r2, lat_r2, ssh_r2,
        transform=ccrs.PlateCarree(),
        cmap=palette,
        vmin=-1, vmax=1,
        shading="auto")

plt.subplots_adjust(right=0.85,left=0.01,top=0.99,bottom=0.1)
cbar_ax = fig.add_axes([0.9, 0.1, 0.04, 0.7])
h=plt.colorbar(pp, cax=cbar_ax,orientation='vertical')
h.set_label('SSHA Karin unsmoothed corrected and flagged (m)',fontsize=14)
h.ax.tick_params(labelsize=14)

/home/jpluser/miniforge3/envs/jupyter/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:315: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return func(self, *args, **kwargs)
/home/jpluser/miniforge3/envs/jupyter/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:315: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return func(self, *args, **kwargs)
/home/jpluser/miniforge3/envs/jupyter/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:315: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return func(self, *args, **kwargs)
/home/jpluser/miniforge3/envs/jupyter/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:315: UserWarning: The colormap's 'bad' has been set, but in order to wrap pcolormesh across the map it must be fully transparent.
  return 